# Импрорты и нужные классы

In [ ]:
import sys
import os
import torch
import copy
import numpy as np
from dotenv import load_dotenv
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".env").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
ENV_PATH = os.path.join(PROJECT_ROOT, ".env")

BASE_DIR = PROJECT_ROOT / "experiments" / "data_chunks_exp_top5"

# sys.path.append('..')
# env_file = os.path.join('..', '.env')
load_dotenv(ENV_PATH)
rowGlam_type = os.environ['ROW_GLAM_TYPE'] # custom or base
name_dataset = os.environ['NAME_DATASET']
name_test_dataset = os.environ['NAME_TEST_DATASET']
dataset_path = os.environ['DATASET_PATH']
test_path = os.environ['TEST_PATH']
test_coco_path = os.environ['TEST_COCO_PATH']
coco_path = os.environ['COCO_PATH']
cache_pdf = os.environ['CASH_PDF_PATH']

In [ ]:
from rows2regionsGLAM.utils.pdf_manager import PDFManager
from rows2regionsGLAM.utils.loger import Loger
from rows2regionsGLAM.utils.row_manager import RowManager
from rows2regionsGLAM.utils.ploter import Ploter
from rows2regionsGLAM.utils.trainer import Trainer
from rows2regionsGLAM.utils.tester import Tester
from rows2regionsGLAM.utils.cacher import Cacher
from rows2regionsGLAM.utils.coco_manager import COCOManager
from rows2regionsGLAM.utils.imbalance import calculate_imbalance
from rows2regionsGLAM.utils.tester import collect_maps, print_map_table

from rows2regionsGLAM.models.rowGLAM_base import TorchModelBase, PARAMS_BASE
from rows2regionsGLAM.models.rowGLAM_custom import TorchModel, PARAMS

from rows2regionsGLAM.tokenizers import RowGLAMTokenizer
from rows2regionsGLAM.converters import Rows2Regions
from rows2regionsGLAM.datasetloaders.base_line_dataset import GLAMDataset
from pager.page_model.sub_models.dtype import ImageSegment
from pager.page_model.sub_models import RegionModel, RowsModel

In [ ]:
loger = Loger(os.path.join(BASE_DIR, 'log.txt'))
pdf_manager = PDFManager(conf={"loger": loger, "pdf_reader": "PDFMiner"})
row_manager = RowManager(conf={"loger": loger, "add_image": True})
coco_manager = COCOManager(conf={"loger": loger, "coco_path": coco_path})
ploter = Ploter(conf={"loger": loger})
tokenizer = RowGLAMTokenizer()
loger(tokenizer.get_name())

pdf2torch_dict = Cacher({
    "loger": loger,
    "pdf_manager": pdf_manager,
    "row_manager": row_manager,
    "tokenizer": tokenizer
})

# Проверка чтения файла

In [ ]:
json_true_regions, CLASSES = coco_manager.get_regions_from_json()

In [ ]:
CLASSES[3] = "text"
CLASSES

In [ ]:
file_names = list(json_true_regions.keys())
file_names.sort()

In [ ]:
num_file = 1
file_name = file_names[num_file]
dataset_file = os.path.join(dataset_path, file_name)
pdf_json, pdf_img = pdf_manager.get_json_and_img_from_pdf(dataset_file, num_page=0)
row_json = row_manager.get_row_json_from_pdf_json(pdf_json)
torch_dict = tokenizer(row_json, pdf_img)
ploter.set_dpi(200)
ploter.plot_img(pdf_img)
ploter.plot_tokens(tokenizer, torch_dict)

# Сохранение датасета

In [ ]:
dataset = GLAMDataset({
    "loger": loger,
    "pdf_dir": dataset_path,
    "coco_file": coco_path,
    "count_class": len(CLASSES),
    "name_dataset": name_dataset,
    "default_index": 0,
    "cache_dir": cache_pdf,
    "pdf2torch_dict": pdf2torch_dict, 
    "to_ROM": True
})

# Создание Cache
N = len(dataset)
for i, d in enumerate(dataset):
    print(f"{(i+1)/N*100:4.2f} %", end='\r')

In [ ]:
test_dataset = GLAMDataset(
    {
    "loger": loger,
    "pdf_dir": test_path,
    "coco_file": test_coco_path,
    "count_class": len(CLASSES),
    "name_dataset": name_dataset,
    "default_index": 0,
    "cache_dir": cache_pdf,
    "pdf2torch_dict": pdf2torch_dict,
    "to_ROM": True
    }
)

N = len(test_dataset)
for i, d in enumerate(test_dataset):
    print(f"{(i+1)/N*100:4.2f} %", end='\r')

# Отрисовска файла из датасета

In [ ]:
torch_dict2 = dataset[100]
path_pdf = os.path.join(dataset_path, torch_dict2['file_name'] + ".pdf")
_, img = pdf_manager.get_json_and_img_from_pdf(path_pdf)
ploter.set_dpi(200)
ploter.plot_img(img)
ploter.plot_tokens(tokenizer, torch_dict2, markup=True)

# Параметры эксперимента

In [ ]:
batch_and_lr = [(8, 0.001), (16, 0.001), (16, 0.0001), (32, 0.001), (8, 0.0001)]

# Вспомогательные функции для сбора метрик

In [ ]:
import re
import matplotlib.pyplot as plt
import json
def parse_log(log_path: Path):
    train_losses = []
    val_losses = []

    map_score = None
    f1_row_95 = None

    epoch_pattern = re.compile(
        r"EPOCH\s+#(\d+)\s+([\d.]+)\s+\(VAL:\s+([\d.]+)\)"
    )

    map_pattern = re.compile(
        r"mAP@IoU\[0\.50:0\.95\]\s+:(\d+\.\d+)"
    )

    f1_95_pattern = re.compile(
        r"threshold_95.*?f1_row\s+:(\d+\.\d+)",
        re.DOTALL
    )

    text = log_path.read_text()

    for match in epoch_pattern.finditer(text):
        train_losses.append(float(match.group(2)))
        val_losses.append(float(match.group(3)))

    map_match = map_pattern.search(text)
    if map_match:
        map_score = float(map_match.group(1))

    f1_match = f1_95_pattern.search(text)
    if f1_match:
        f1_row_95 = float(f1_match.group(1))

    return np.array(train_losses), np.array(val_losses), map_score, f1_row_95

def analyze_curves(train_losses, val_losses):
    best_train_epoch = int(np.argmin(train_losses))

    diff = train_losses - val_losses

    sign_change_idx = np.where(np.diff(np.sign(diff)) != 0)[0]

    if len(sign_change_idx) > 0:
        intersection_epoch = int(sign_change_idx[0] + 1)
        intersection_type = "sign_change"
    else:
        intersection_epoch = int(np.argmin(np.abs(diff)))
        intersection_type = "min_distance"

    return best_train_epoch, intersection_epoch, intersection_type


def plot_losses(train_losses, val_losses, intersection_epoch, save_path: Path):
    plt.figure(figsize=(8, 5))

    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Validation Loss")

    plt.axvline(intersection_epoch, linestyle="--", label="Intersection")

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training Curve")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# Запуск эксперимента

In [ ]:
for batch_size, learning_rate in batch_and_lr:
    model_name = f'row2region_GLAM_batch{batch_size}_lr{str(learning_rate)[2:]}'
    
    if rowGlam_type == "base":
        model_params = copy.deepcopy(PARAMS_BASE)
    elif rowGlam_type == "custom":
        model_params = copy.deepcopy(PARAMS)
        
    model_params["NodeClasses"] = len(CLASSES)
    
    model_params["epochs"] = 30
    model_params["batch_size"] = batch_size
    model_params["learning_rate"] = learning_rate
    
    # Дисбаланс классов ------------------------------------
    publaynet_imbalance, edge_imbalance = calculate_imbalance(dataset)
    
    model_params['loss_params']['publaynet_imbalance'] = publaynet_imbalance
    model_params['loss_params']['edge_imbalance'] = edge_imbalance

    # Обучение модели ------------------------------------
    exp_name = f"batch{batch_size}_lr{str(learning_rate)[2:]}"
    exp_dir = BASE_DIR / exp_name
    exp_dir.mkdir(parents=True)
    log_path = Path(os.path.join(exp_dir, 'log.txt'))
    loger_exp = Loger(log_path)
    
    trainer_pub = Trainer(conf={"loger": loger_exp, "params": model_params, "model_name": os.path.join(exp_dir, model_name)})
    trainer_pub.start_train(5, dataset)

    # Тестирование ------------------------------------
    model_params['sigmoidEdge'] = True
    if rowGlam_type == "base":
        model = TorchModelBase(model_params)
    elif rowGlam_type == "custom":
        model = TorchModel(model_params)
    
    model.load_state_dict(torch.load(os.path.join(exp_dir, model_name), weights_only=True))
    
    rows_model = RowsModel()
    region_model = RegionModel()
    rows2regions = Rows2Regions({
        'model':model, 
        'tokenizer': RowGLAMTokenizer(),
        'is_merge_extract': True,
        'classes': CLASSES
    })

    tester = Tester(conf={
        "loger": loger_exp, 
        "pdf_manager": pdf_manager, 
        "row_manager": row_manager, 
        "rows_model": rows_model, 
        "region_model": region_model, 
        "rows2regions": rows2regions})
    
    metrics = tester.calculate_target_and_preds(
        test_dataset, 
        name_dataset, 
        name_test_dataset, 
        dataset_path, 
        test_path
    )
    tester.print_result(metrics)
    
    train_losses, val_losses, map_score, f1_row_95 = parse_log(log_path)
    
    best_train_epoch, intersection_epoch, intersection_type = analyze_curves(train_losses, val_losses)
    
    
    plot_path = exp_dir / "loss_curve.png"
    plot_losses(train_losses, val_losses, intersection_epoch, plot_path)
    
    metrics = {
        "mAP_50_95": map_score,
        "f1_row_threshold_95": f1_row_95,
        "best_train_epoch": best_train_epoch,
        "best_train_loss": float(train_losses[best_train_epoch]),
    
        "intersection_epoch": intersection_epoch,
        "intersection_type": intersection_type,
    
        "best_val_loss": float(np.min(val_losses)),
        "best_val_epoch": int(np.argmin(val_losses)),
    }
    
    metrics_path = exp_dir / "metrics.json"
    with open(metrics_path, "w") as f:
        json.dump(metrics, f, indent=4)